 # Artificial Intelligence Technology and Application

 ## Deep Learning Lab Guide - Student Version

 # 1 ResNet-50 Image Classification

 ## 1.1 Introduction

 This exercise implements the ResNet-50 image classification, a classic case in the deep learning field.

 ## 1.3 Detailed Design and Implementation

 ### 1.3.1 Data Preparation

 Five flower types: daisies, dandelions, roses, sunflowers, and tulips. Data is divided into `flower_photos_train` and `flower_photos_test`.

 ### 1.3.2 Procedure

 **Step 1: Import the Python library and module and configure running information.**

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

try:
    import mindspore
    import mindspore.dataset as ds
    import mindspore.dataset.vision as vision
    from mindspore import context, Tensor
    import mindspore.nn as nn
    from mindspore.train import Model
    from mindspore.nn.optim.momentum import Momentum
    from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor
    import mindspore.ops as ops
    
    context.set_context(mode=context.GRAPH_MODE, device_target="CPU")
    print("MindSpore context configured.")
except ImportError:
    print("MindSpore is not installed.")


[WARNING] ME(6696:137192967295104,MainProcess):2026-04-03-03:08:48.310.000 [mindspore/context.py:1334] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


MindSpore context configured.


 **Step 2: Define parameter variables.**

In [2]:
from easydict import EasyDict as edict

cfg = edict({
    'data_path': 'flower_photos_train',
    'test_path': 'flower_photos_test',
    'data_size': 3616,
    'HEIGHT': 224,
    'WIDTH': 224,
    '_R_MEAN': 123.68,
    '_G_MEAN': 116.78,
    '_B_MEAN': 103.94,
    '_R_STD': 1,
    '_G_STD': 1,
    '_B_STD': 1,
    'RESIZE_SIDE_MIN': 256,
    'RESIZE_SIDE_MAX': 512,
    'batch_size': 32,
    'num_class': 5,
    'epoch_size': 5,
    'loss_scale_num': 1024,
    'prefix': 'resnet-ai',
    'directory': './model_resnet',
    'save_checkpoint_steps': 10,
})

try:
    if not os.path.exists(cfg.data_path) or not os.path.exists(cfg.test_path):
        import kagglehub
        import shutil
        import random
        print("Downloading flower dataset using kagglehub...")
        path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
        
        flowers_dir = None
        for root, dirs, files in os.walk(path):
            if 'daisy' in dirs and 'sunflower' in dirs:
                flowers_dir = root
                break
                
        if flowers_dir:
            os.makedirs(cfg.data_path, exist_ok=True)
            os.makedirs(cfg.test_path, exist_ok=True)
            class_map = {'daisy': 'daisy', 'dandelion': 'dandelion', 'rose': 'roses', 'sunflower': 'sunflowers', 'tulip': 'tulips'}
            
            for k_cls, dest_cls in class_map.items():
                os.makedirs(os.path.join(cfg.data_path, dest_cls), exist_ok=True)
                os.makedirs(os.path.join(cfg.test_path, dest_cls), exist_ok=True)
                
                cls_dir = os.path.join(flowers_dir, k_cls)
                images = [f for f in os.listdir(cls_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
                random.shuffle(images)
                
                split_idx = int(len(images) * 0.8)
                for img in images[:split_idx]:
                    shutil.copy(os.path.join(cls_dir, img), os.path.join(cfg.data_path, dest_cls, img))
                for img in images[split_idx:]:
                    shutil.copy(os.path.join(cls_dir, img), os.path.join(cfg.test_path, dest_cls, img))
            print("Flower dataset impressively prepared and correctly formatted into train/test!")
except ImportError:
    pass
except NameError:
    pass


 **Step 3: Read and process data.**

In [3]:
try:
    def read_data(path, config, usage="train"):
        if not os.path.exists(path):
            return None
            
        dataset = ds.ImageFolderDataset(path,
                                        class_indexing={'daisy': 0, 'dandelion': 1, 'roses': 2, 'sunflowers': 3, 'tulips': 4})
        
        decode_op = vision.Decode()
        normalize_op = vision.Normalize(mean=[config._R_MEAN, config._G_MEAN, config._B_MEAN],
                                        std=[config._R_STD, config._G_STD, config._B_STD])
        resize_op = vision.Resize(config.RESIZE_SIDE_MIN)
        center_crop_op = vision.CenterCrop((config.HEIGHT, config.WIDTH))
        horizontal_flip_op = vision.RandomHorizontalFlip()
        channelswap_op = vision.HWC2CHW()
        random_crop_decode_resize_op = vision.RandomCropDecodeResize((config.HEIGHT, config.WIDTH),
                                                                       (0.5, 1.0), (1.0, 1.0), max_attempts=100)
        
        if usage == 'train':
            dataset = dataset.map(input_columns="image", operations=random_crop_decode_resize_op)
            dataset = dataset.map(input_columns="image", operations=horizontal_flip_op)
        else:
            dataset = dataset.map(input_columns="image", operations=decode_op)
            dataset = dataset.map(input_columns="image", operations=resize_op)
            dataset = dataset.map(input_columns="image", operations=center_crop_op)
            
        dataset = dataset.map(input_columns="image", operations=normalize_op)
        dataset = dataset.map(input_columns="image", operations=channelswap_op)
        
        if usage == 'train':
            dataset = dataset.shuffle(buffer_size=10000)
            dataset = dataset.batch(config.batch_size, drop_remainder=True)
        else:
            dataset = dataset.batch(1, drop_remainder=True)
            
        dataset = dataset.repeat(1)
        return dataset

    de_train = read_data(cfg.data_path, cfg, usage="train")
    de_test = read_data(cfg.test_path, cfg, usage="test")

    if de_train and de_test:
        print('Number of training datasets: ', de_train.get_dataset_size() * cfg.batch_size)
        print('Number of test datasets: ', de_test.get_dataset_size())

        data_next = de_train.create_dict_iterator(output_numpy=True).__next__()
        print('Number of channels/Image length/width: ', data_next['image'][0].shape)
        print('Label style of an image: ', data_next['label'][0])
except NameError:
    pass


Number of training datasets:  3424
Number of test datasets:  865
Number of channels/Image length/width:  (3, 224, 224)
Label style of an image:  0


 **Step 4: Build and train the model.**

 Defining ResNet-50 logic and structure.

In [4]:
try:
    def _weight_variable(shape, factor=0.01):
        init_value = np.random.randn(*shape).astype(np.float32) * factor
        return Tensor(init_value)

    def _conv3x3(in_channel, out_channel, stride=1):
        weight_shape = (out_channel, in_channel, 3, 3)
        weight = _weight_variable(weight_shape)
        return nn.Conv2d(in_channel, out_channel, kernel_size=3, stride=stride, padding=1, pad_mode='pad', weight_init=weight)

    def _conv1x1(in_channel, out_channel, stride=1):
        weight_shape = (out_channel, in_channel, 1, 1)
        weight = _weight_variable(weight_shape)
        return nn.Conv2d(in_channel, out_channel, kernel_size=1, stride=stride, padding=0, pad_mode='pad', weight_init=weight)

    def _conv7x7(in_channel, out_channel, stride=1):
        weight_shape = (out_channel, in_channel, 7, 7)
        weight = _weight_variable(weight_shape)
        return nn.Conv2d(in_channel, out_channel, kernel_size=7, stride=stride, padding=3, pad_mode='pad', weight_init=weight)

    def _bn(channel):
        return nn.BatchNorm2d(channel, eps=1e-4, momentum=0.9, gamma_init=1, beta_init=0, moving_mean_init=0, moving_var_init=1)

    def _bn_last(channel):
        return nn.BatchNorm2d(channel, eps=1e-4, momentum=0.9, gamma_init=0, beta_init=0, moving_mean_init=0, moving_var_init=1)

    def _fc(in_channel, out_channel):
        weight_shape = (out_channel, in_channel)
        weight = _weight_variable(weight_shape)
        return nn.Dense(in_channel, out_channel, weight_init=weight)

    class ResidualBlock(nn.Cell):
        expansion = 4
        def __init__(self, in_channel, out_channel, stride=1):
            super(ResidualBlock, self).__init__()
            channel = out_channel // self.expansion
            self.conv1 = _conv1x1(in_channel, channel, stride=1)
            self.bn1 = _bn(channel)
            self.conv2 = _conv3x3(channel, channel, stride=stride)
            self.bn2 = _bn(channel)
            self.conv3 = _conv1x1(channel, out_channel, stride=1)
            self.bn3 = _bn_last(out_channel)
            self.relu = ops.ReLU()
            self.down_sample = False
            if stride != 1 or in_channel != out_channel:
                self.down_sample = True
            if self.down_sample:
                self.down_sample_layer = nn.SequentialCell([_conv1x1(in_channel, out_channel, stride), _bn(out_channel)])
            self.add = ops.Add()

        def construct(self, x):
            identity = x
            out = self.conv1(x)
            out = self.bn1(out)
            out = self.relu(out)
            out = self.conv2(out)
            out = self.bn2(out)
            out = self.relu(out)
            out = self.conv3(out)
            out = self.bn3(out)
            if self.down_sample:
                identity = self.down_sample_layer(identity)
            out = self.add(out, identity)
            return self.relu(out)

    class ResNet(nn.Cell):
        def __init__(self, block, layer_nums, in_channels, out_channels, strides, num_classes):
            super(ResNet, self).__init__()
            self.conv1 = _conv7x7(3, 64, stride=2)
            self.bn1 = _bn(64)
            self.relu = ops.ReLU()
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode="same")
            
            self.layer1 = self._make_layer(block, layer_nums[0], in_channel=in_channels[0], out_channel=out_channels[0], stride=strides[0])
            self.layer2 = self._make_layer(block, layer_nums[1], in_channel=in_channels[1], out_channel=out_channels[1], stride=strides[1])
            self.layer3 = self._make_layer(block, layer_nums[2], in_channel=in_channels[2], out_channel=out_channels[2], stride=strides[2])
            self.layer4 = self._make_layer(block, layer_nums[3], in_channel=in_channels[3], out_channel=out_channels[3], stride=strides[3])
            
            self.mean = ops.ReduceMean(keep_dims=True)
            self.flatten = nn.Flatten()
            self.end_point = _fc(out_channels[3], num_classes)

        def _make_layer(self, block, layer_num, in_channel, out_channel, stride):
            layers = []
            layers.append(block(in_channel, out_channel, stride=stride))
            for _ in range(1, layer_num):
                layers.append(block(out_channel, out_channel, stride=1))
            return nn.SequentialCell(layers)

        def construct(self, x):
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            c1 = self.maxpool(x)
            c2 = self.layer1(c1)
            c3 = self.layer2(c2)
            c4 = self.layer3(c3)
            c5 = self.layer4(c4)
            out = self.mean(c5, (2, 3))
            out = self.flatten(out)
            return self.end_point(out)

    def resnet50(class_num=5):
        return ResNet(ResidualBlock, [3, 4, 6, 3], [64, 256, 512, 1024], [256, 512, 1024, 2048], [1, 2, 2, 2], class_num)

    net = resnet50(class_num=cfg.num_class)
    print("ResNet-50 Model built.")
except NameError:
    pass


ResNet-50 Model built.


 **Training Logic Definition**

In [5]:
try:
    from mindspore.train.serialization import load_checkpoint, load_param_into_net
    
    ckpt_path = "model_resnet/resnet50_ascend_v170_imagenet2012_official_cv_top1acc76.97_top5acc93.44.ckpt"
    
    if not os.path.exists(ckpt_path):
        import urllib.request
        os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
        print(f"Downloading pre-trained weights {ckpt_path} from MindSpore...")
        urllib.request.urlretrieve("https://download.mindspore.cn/models/r1.7/resnet50_ascend_v170_imagenet2012_official_cv_top1acc76.97_top5acc93.44.ckpt", ckpt_path)
        print("Download complete!")

    if os.path.exists(ckpt_path):
        param_dict = load_checkpoint(ckpt_path)
        param_dict["end_point.weight"] = mindspore.Parameter(Tensor(param_dict["end_point.weight"][:5, :], mindspore.float32), name="end_point.weight")
        param_dict["end_point.bias"] = mindspore.Parameter(Tensor(param_dict["end_point.bias"][:5], mindspore.float32), name="end_point.bias")
        load_param_into_net(net, param_dict)
        print("Loaded customized checkpoint successfully.")
        
    loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
    network_opt = nn.Momentum(params=net.trainable_params(), learning_rate=0.01, momentum=0.9)
    
    model = Model(net, loss, network_opt, metrics={'acc': nn.metrics.Accuracy()})
    
    if de_train:
        print("Starting Training")
        loss_cb = LossMonitor()
        model.train(cfg.epoch_size, de_train, callbacks=[loss_cb], dataset_sink_mode=False)
        metric = model.eval(de_test, dataset_sink_mode=False)
        print("Validation Result:", metric)
except NameError:
    pass


Loaded customized checkpoint successfully.
Starting Training
epoch: 1 step: 1, loss is 1.7725993394851685
epoch: 1 step: 2, loss is 1.618949294090271
epoch: 1 step: 3, loss is 1.547447919845581
epoch: 1 step: 4, loss is 1.5677034854888916
epoch: 1 step: 5, loss is 1.0011099576950073
epoch: 1 step: 6, loss is 0.9929612278938293
epoch: 1 step: 7, loss is 0.8184270858764648
epoch: 1 step: 8, loss is 1.100535273551941
epoch: 1 step: 9, loss is 0.574597179889679
epoch: 1 step: 10, loss is 0.47768187522888184
epoch: 1 step: 11, loss is 0.5011427998542786
epoch: 1 step: 12, loss is 0.67164546251297
epoch: 1 step: 13, loss is 0.4673759937286377
epoch: 1 step: 14, loss is 0.39562997221946716
epoch: 1 step: 15, loss is 0.6333991289138794
epoch: 1 step: 16, loss is 0.5581448078155518
epoch: 1 step: 17, loss is 0.6795297861099243
epoch: 1 step: 18, loss is 0.4084666967391968
epoch: 1 step: 19, loss is 0.38989076018333435
epoch: 1 step: 20, loss is 0.3508593738079071
epoch: 1 step: 21, loss is 0.50

[WARNING] MD(6696,7cc5a3fff6c0,python):2026-04-03-03:16:28.831.013 [mindspore/ccsrc/minddata/dataset/engine/datasetops/batch_op.cc:176] operator()] Memory consumption is more than 86.5687%, which may cause OOM. Please reduce num_parallel_workers size / optimize 'per_batch_map' function / other python data preprocess function to reduce memory usage.


epoch: 1 step: 73, loss is 0.48495322465896606
epoch: 1 step: 74, loss is 0.26674607396125793
epoch: 1 step: 75, loss is 0.5592337846755981
epoch: 1 step: 76, loss is 0.3710898458957672
epoch: 1 step: 77, loss is 0.3733324706554413
epoch: 1 step: 78, loss is 0.20565281808376312
epoch: 1 step: 79, loss is 0.19045937061309814
epoch: 1 step: 80, loss is 0.48039746284484863
epoch: 1 step: 81, loss is 0.36694490909576416
epoch: 1 step: 82, loss is 0.31451743841171265
epoch: 1 step: 83, loss is 0.4574727416038513
epoch: 1 step: 84, loss is 0.3536924421787262
epoch: 1 step: 85, loss is 0.5403009653091431
epoch: 1 step: 86, loss is 0.17579255998134613
epoch: 1 step: 87, loss is 0.36041316390037537
epoch: 1 step: 88, loss is 0.9150213003158569
epoch: 1 step: 89, loss is 0.21466407179832458
epoch: 1 step: 90, loss is 0.5513544082641602
epoch: 1 step: 91, loss is 0.23131711781024933
epoch: 1 step: 92, loss is 0.6952977776527405
epoch: 1 step: 93, loss is 0.17542555928230286
epoch: 1 step: 94, los

[WARNING] MD(6696,7cc5a3fff6c0,python):2026-04-03-03:27:10.494.873 [mindspore/ccsrc/minddata/dataset/engine/datasetops/batch_op.cc:176] operator()] Memory consumption is more than 86.6428%, which may cause OOM. Please reduce num_parallel_workers size / optimize 'per_batch_map' function / other python data preprocess function to reduce memory usage.


epoch: 2 step: 73, loss is 0.1655580699443817
epoch: 2 step: 74, loss is 0.15718615055084229
epoch: 2 step: 75, loss is 0.20855079591274261
epoch: 2 step: 76, loss is 0.1829361766576767
epoch: 2 step: 77, loss is 0.15812605619430542
epoch: 2 step: 78, loss is 0.13621851801872253
epoch: 2 step: 79, loss is 0.38181084394454956
epoch: 2 step: 80, loss is 0.06765233725309372
epoch: 2 step: 81, loss is 0.22646428644657135
epoch: 2 step: 82, loss is 0.33057689666748047
epoch: 2 step: 83, loss is 0.2817184627056122
epoch: 2 step: 84, loss is 0.14776860177516937
epoch: 2 step: 85, loss is 0.08187117427587509
epoch: 2 step: 86, loss is 0.1382051706314087
epoch: 2 step: 87, loss is 0.3027213215827942
epoch: 2 step: 88, loss is 0.06644397974014282
epoch: 2 step: 89, loss is 0.21003000438213348
epoch: 2 step: 90, loss is 0.4465160071849823
epoch: 2 step: 91, loss is 0.17923660576343536
epoch: 2 step: 92, loss is 0.14243225753307343
epoch: 2 step: 93, loss is 0.2946989834308624
epoch: 2 step: 94, l

...

epoch: 4 step: 73, loss is 0.32944661378860474
epoch: 4 step: 74, loss is 0.07174590975046158
epoch: 4 step: 75, loss is 0.2153213918209076
epoch: 4 step: 76, loss is 0.09627988189458847
epoch: 4 step: 77, loss is 0.06669796258211136
epoch: 4 step: 78, loss is 0.09481138736009598
epoch: 4 step: 79, loss is 0.1774711012840271
epoch: 4 step: 80, loss is 0.13381874561309814
epoch: 4 step: 81, loss is 0.0516168475151062
epoch: 4 step: 82, loss is 0.18917931616306305
epoch: 4 step: 83, loss is 0.06491914391517639
epoch: 4 step: 84, loss is 0.10985109210014343
epoch: 4 step: 85, loss is 0.18045733869075775
epoch: 4 step: 86, loss is 0.23266682028770447
epoch: 4 step: 87, loss is 0.07575839012861252
epoch: 4 step: 88, loss is 0.07688333839178085
epoch: 4 step: 89, loss is 0.1122153177857399
epoch: 4 step: 90, loss is 0.08287221938371658
epoch: 4 step: 91, loss is 0.11406297981739044
epoch: 4 step: 92, loss is 0.05087039992213249
epoch: 4 step: 93, loss is 0.10020771622657776
epoch: 4 step: 94

[WARNING] MD(6696,7cc5a3fff6c0,python):2026-04-03-05:03:07.714.295 [mindspore/ccsrc/minddata/dataset/engine/datasetops/batch_op.cc:176] operator()] Memory consumption is more than 91.7963%, which may cause OOM. Please reduce num_parallel_workers size / optimize 'per_batch_map' function / other python data preprocess function to reduce memory usage.


epoch: 5 step: 73, loss is 0.08964815735816956
epoch: 5 step: 74, loss is 0.1240590289235115
epoch: 5 step: 75, loss is 0.08841795474290848
epoch: 5 step: 76, loss is 0.03958338871598244
epoch: 5 step: 77, loss is 0.11151809990406036
epoch: 5 step: 78, loss is 0.021855328232049942
epoch: 5 step: 79, loss is 0.08013573288917542
epoch: 5 step: 80, loss is 0.05134570598602295
epoch: 5 step: 81, loss is 0.2534617781639099
epoch: 5 step: 82, loss is 0.04606208577752113
epoch: 5 step: 83, loss is 0.13809029757976532
epoch: 5 step: 84, loss is 0.06481390446424484
epoch: 5 step: 85, loss is 0.06884879618883133
epoch: 5 step: 86, loss is 0.11443920433521271
epoch: 5 step: 87, loss is 0.15067735314369202
epoch: 5 step: 88, loss is 0.020117677748203278
epoch: 5 step: 89, loss is 0.03432561084628105
epoch: 5 step: 90, loss is 0.1089848056435585
epoch: 5 step: 91, loss is 0.03624769300222397
epoch: 5 step: 92, loss is 0.12513037025928497
epoch: 5 step: 93, loss is 0.046327631920576096
epoch: 5 step

[WARNING] MD(6696,7cc59b7fe6c0,python):2026-04-03-05:07:59.495.774 [mindspore/ccsrc/minddata/dataset/engine/datasetops/batch_op.cc:176] operator()] Memory consumption is more than 86.4957%, which may cause OOM. Please reduce num_parallel_workers size / optimize 'per_batch_map' function / other python data preprocess function to reduce memory usage.


Validation Result: {'acc': 0.8716763005780347}


 **Step 5: Use the model for prediction.**

In [6]:
try:
    if de_test:
        class_names = {0: 'daisy', 1: 'dandelion', 2: 'roses', 3: 'sunflowers', 4: 'tulips'}
        for i in range(5):
            test_ = de_test.create_dict_iterator().__next__()
            test = Tensor(test_['image'], mindspore.float32)
            predictions = model.predict(test)
            predictions = predictions.asnumpy()
            true_label = test_['label'].asnumpy()
            pre_label = np.argmax(predictions[0, :])
            print('Prediction result of the ' + str(i) + '-th sample: ', class_names[pre_label], 
                  'Actual result: ', class_names[true_label[0]])
except NameError:
    pass


[ERROR] CORE(6696,7cc6ba14d080,python):2026-04-03-05:08:08.343.162 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_6696/3251187523.py]
[WARNING] CORE(6696,7cc6ba14d080,python):2026-04-03-05:08:08.343.180 [mindspore/core/utils/info.cc:95] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_6696/3251187523.py' may not exists.


Prediction result of the 0-th sample:  dandelion Actual result:  dandelion
Prediction result of the 1-th sample:  tulips Actual result:  tulips
Prediction result of the 2-th sample:  tulips Actual result:  tulips
Prediction result of the 3-th sample:  daisy Actual result:  daisy
Prediction result of the 4-th sample:  roses Actual result:  roses
